# HotpotQA BGE Embedding Shards on Kaggle T4

This notebook creates embedding shards compatible with `vdt-meeting-search` Sprint 3. It streams `BeIR/hotpotqa` from Hugging Face, embeds `title + "\n" + text` with `BAAI/bge-small-en-v1.5`, writes float16 vectors and uint64 numeric ids, then packs outputs for download.

Recommended accelerator: **GPU T4 x2** if available. Dual GPU is handled by splitting shard ranges across two independent worker processes, one per GPU. This is safer and usually more effective for corpus embedding than `DataParallel`.


## Kaggle Setup

In Kaggle Notebook settings, choose:

- Accelerator: `GPU T4 x2` if available, otherwise `GPU T4`
- Internet: On

If only one GPU appears, set `USE_DUAL_GPU = False`.


In [ ]:
!nvidia-smi
!python -m pip install -q --upgrade sentence-transformers==3.0.1 datasets numpy tqdm


## Configuration

If you want Kaggle to embed the full corpus from scratch, keep `START_SHARD = 0`, `END_SHARD = 104`.

If your local machine already completed shards and you only want Kaggle to do the remainder, set `START_SHARD` to the first missing shard number. For example, if local has `docs-00000.done` through `docs-00034.done`, use `START_SHARD = 35`.


In [ ]:
from pathlib import Path

DATASET = 'BeIR/hotpotqa'
NAME = 'corpus'
SPLIT = 'corpus'
MODEL_NAME = 'BAAI/bge-small-en-v1.5'
DOCS_PER_SHARD = 50_000
TOTAL_DOCS = 5_233_329
TOTAL_SHARDS = 105

# Edit these if you only want Kaggle to produce a subset/range.
START_SHARD = 35
END_SHARD = 104

# T4 has 16GB VRAM, so 256 is usually worth trying. Lower to 128 if OOM.
BATCH_SIZE = 256

# If Kaggle session has two GPUs, this runs two independent shard workers.
USE_DUAL_GPU = True

WORK_DIR = Path('/kaggle/working/hotpotqa_bge')
EMBEDDING_DIR = WORK_DIR / 'embeddings'
PROGRESS_DIR = WORK_DIR / 'progress' / 'embed'
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', WORK_DIR)


In [ ]:
%%writefile /kaggle/working/embed_hotpotqa_kaggle_worker.py
from __future__ import annotations

import argparse
import json
import os
import time
from pathlib import Path

import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer


def collapse(value: str) -> str:
    return ' '.join(str(value or '').split())


def content_from_row(row: dict) -> str:
    title = collapse(row.get('title', '') or '')
    text = collapse(row.get('text', '') or '')
    return title + '\n' + text if title and text else title or text


def done_marker(progress_dir: Path, shard_idx: int) -> Path:
    return progress_dir / f'docs-{shard_idx:05d}.done'


def save_shard(shard_idx: int, texts: list[str], ids: list[int], embedding_dir: Path, model, batch_size: int, model_name: str) -> dict:
    start = time.perf_counter()
    vectors = model.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        batch_size=batch_size,
        show_progress_bar=False,
    )
    vectors = np.asarray(vectors, dtype=np.float32)
    ids_array = np.asarray(ids, dtype=np.uint64)
    stem = f'docs-{shard_idx:05d}'
    np.save(embedding_dir / f'{stem}.float16.npy', vectors.astype(np.float16))
    np.save(embedding_dir / f'{stem}.ids.npy', ids_array)
    meta = {
        'source_file': f'{stem}.jsonl',
        'docs': int(len(ids)),
        'dims': int(vectors.shape[1]),
        'model': model_name,
        'numeric_id_start': int(ids[0]) if ids else None,
        'numeric_id_end': int(ids[-1]) if ids else None,
        'elapsed_seconds': round(time.perf_counter() - start, 3),
    }
    (embedding_dir / f'{stem}.meta.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
    return meta


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument('--dataset', default='BeIR/hotpotqa')
    parser.add_argument('--name', default='corpus')
    parser.add_argument('--split', default='corpus')
    parser.add_argument('--model', default='BAAI/bge-small-en-v1.5')
    parser.add_argument('--embedding-dir', type=Path, required=True)
    parser.add_argument('--progress-dir', type=Path, required=True)
    parser.add_argument('--docs-per-shard', type=int, default=50_000)
    parser.add_argument('--total-shards', type=int, default=105)
    parser.add_argument('--start-shard', type=int, required=True)
    parser.add_argument('--end-shard', type=int, required=True)
    parser.add_argument('--batch-size', type=int, default=256)
    parser.add_argument('--device', default='cuda:0')
    args = parser.parse_args()

    args.embedding_dir.mkdir(parents=True, exist_ok=True)
    args.progress_dir.mkdir(parents=True, exist_ok=True)
    print(f'[worker] pid={os.getpid()} device={args.device} shards={args.start_shard}-{args.end_shard} batch={args.batch_size}', flush=True)
    model = SentenceTransformer(args.model, device=args.device)
    print(f'[worker] model_device={model.device}', flush=True)

    stream = load_dataset(args.dataset, args.name, split=args.split, streaming=True)
    target_start_doc = args.start_shard * args.docs_per_shard
    target_end_doc = ((args.end_shard + 1) * args.docs_per_shard) - 1

    shard_texts: list[str] = []
    shard_ids: list[int] = []
    current_shard = None
    completed = 0
    run_start = time.perf_counter()

    for numeric_id, row in enumerate(stream):
        if numeric_id < target_start_doc:
            continue
        if numeric_id > target_end_doc:
            break
        shard_idx = numeric_id // args.docs_per_shard
        if shard_idx < args.start_shard or shard_idx > args.end_shard:
            continue
        if done_marker(args.progress_dir, shard_idx).exists():
            current_shard = shard_idx
            continue
        if current_shard is None:
            current_shard = shard_idx
            print(f'[worker] start docs-{current_shard:05d}', flush=True)
        if shard_idx != current_shard:
            meta = save_shard(current_shard, shard_texts, shard_ids, args.embedding_dir, model, args.batch_size, args.model)
            done_marker(args.progress_dir, current_shard).write_text(json.dumps(meta, indent=2), encoding='utf-8')
            completed += 1
            done_count = len(list(args.progress_dir.glob('*.done')))
            elapsed = time.perf_counter() - run_start
            print(
                f'[worker] done docs-{current_shard:05d} docs={meta["docs"]} '
                f'elapsed={meta["elapsed_seconds"]}s global_done={done_count}/{args.total_shards} '
                f'completed_this_worker={completed} wall={elapsed/60:.1f}m',
                flush=True,
            )
            shard_texts = []
            shard_ids = []
            current_shard = shard_idx
            print(f'[worker] start docs-{current_shard:05d}', flush=True)
        shard_texts.append(content_from_row(row))
        shard_ids.append(numeric_id)

    if current_shard is not None and shard_texts and not done_marker(args.progress_dir, current_shard).exists():
        meta = save_shard(current_shard, shard_texts, shard_ids, args.embedding_dir, model, args.batch_size, args.model)
        done_marker(args.progress_dir, current_shard).write_text(json.dumps(meta, indent=2), encoding='utf-8')
        completed += 1
        done_count = len(list(args.progress_dir.glob('*.done')))
        print(
            f'[worker] done docs-{current_shard:05d} docs={meta["docs"]} '
            f'elapsed={meta["elapsed_seconds"]}s global_done={done_count}/{args.total_shards} '
            f'completed_this_worker={completed}',
            flush=True,
        )
    print(f'[worker] finished device={args.device} completed={completed}', flush=True)


if __name__ == '__main__':
    main()


## Run Embedding

For dual T4, the notebook splits the selected shard range into two contiguous ranges and launches one worker per GPU. For single GPU, it runs one worker.


In [ ]:
import subprocess
import sys
import time
import torch

gpu_count = torch.cuda.device_count()
print('gpu_count:', gpu_count)
use_dual = USE_DUAL_GPU and gpu_count >= 2


def worker_cmd(device, start, end):
    return [
        sys.executable, '/kaggle/working/embed_hotpotqa_kaggle_worker.py',
        '--dataset', DATASET, '--name', NAME, '--split', SPLIT,
        '--model', MODEL_NAME,
        '--embedding-dir', str(EMBEDDING_DIR),
        '--progress-dir', str(PROGRESS_DIR),
        '--docs-per-shard', str(DOCS_PER_SHARD),
        '--total-shards', str(TOTAL_SHARDS),
        '--start-shard', str(start), '--end-shard', str(end),
        '--batch-size', str(BATCH_SIZE),
        '--device', device,
    ]


if use_dual:
    mid = (START_SHARD + END_SHARD) // 2
    ranges = [('cuda:0', START_SHARD, mid), ('cuda:1', mid + 1, END_SHARD)]
else:
    ranges = [('cuda:0' if gpu_count else 'cpu', START_SHARD, END_SHARD)]

print('ranges:', ranges)
procs = []
for device, start, end in ranges:
    cmd = worker_cmd(device, start, end)
    print('launch:', ' '.join(cmd))
    procs.append(subprocess.Popen(cmd))

while True:
    done = len(list(PROGRESS_DIR.glob('*.done')))
    pct = round(done * 100 / TOTAL_SHARDS, 2)
    statuses = [p.poll() for p in procs]
    print(time.strftime('%H:%M:%S'), f'progress={done}/{TOTAL_SHARDS} ({pct}%)', 'statuses=', statuses, flush=True)
    if all(status is not None for status in statuses):
        break
    time.sleep(60)

for p in procs:
    if p.returncode != 0:
        raise RuntimeError(f'worker failed with exit code {p.returncode}')
print('embedding workers finished')


## Validate Outputs


In [ ]:
import json
import numpy as np

done_files = sorted(PROGRESS_DIR.glob('*.done'))
vec_files = sorted(EMBEDDING_DIR.glob('*.float16.npy'))
id_files = sorted(EMBEDDING_DIR.glob('*.ids.npy'))
meta_files = sorted(EMBEDDING_DIR.glob('*.meta.json'))
print('done', len(done_files), 'vectors', len(vec_files), 'ids', len(id_files), 'meta', len(meta_files))
if vec_files:
    sample = np.load(vec_files[0])
    sample_ids = np.load(id_files[0])
    print('sample vector file:', vec_files[0].name, sample.shape, sample.dtype)
    print('sample ids:', sample_ids[:3], sample_ids[-3:], sample_ids.dtype)
    print(meta_files[0].read_text())


## Pack For Download

This creates one archive containing embeddings and progress markers. Download it from Kaggle output, then extract into the local repo root.


In [ ]:
archive = Path('/kaggle/working/hotpotqa_bge_embeddings.tar.gz')
!tar -czf {archive} -C /kaggle/working hotpotqa_bge
!ls -lh {archive}


## Local Restore

After downloading `hotpotqa_bge_embeddings.tar.gz`, put it in the local repo root and run:

```powershell
tar -xzf hotpotqa_bge_embeddings.tar.gz
Copy-Item -Recurse -Force hotpotqa_bge\embeddings\* artifacts\hotpotqa_full\embeddings\
Copy-Item -Recurse -Force hotpotqa_bge\progress\embed\* artifacts\hotpotqa_full\progress\embed\
(Get-ChildItem artifacts\hotpotqa_full\progress\embed -Filter *.done).Count
```

Then build TurboVec locally:

```powershell
python scripts/build_turbovec.py --embedding-dir artifacts/hotpotqa_full/embeddings --output artifacts/hotpotqa_full/turbovec/hotpotqa_bge_small_4bit.tvim --config-output artifacts/hotpotqa_full/turbovec/config.json --dim 384 --bit-width 4
```
